# 多ETF轮动+趋势震荡双策略回测

本Notebook展示如何使用重构后的本地量化回测系统。

In [ ]:
import sys
import os

project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
from datetime import datetime

from source import (
    DataLoader, ETFRotationStrategy, BacktestEngine, 
    PerformanceAnalyzer, ETF_POOL, 
    BACKTEST_CONFIG, STRATEGY_CONFIG
)

## 1. 配置回测参数

In [ ]:
config = BACKTEST_CONFIG.copy()
config['start_date'] = '2021-01-01'
config['end_date'] = '2024-06-30'
config['initial_cash'] = 1000000
config['daily_injection'] = 400

print("回测配置:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 2. 加载ETF数据

使用akshare获取真实的ETF历史数据。

In [ ]:
data_loader = DataLoader()

print(f"正在加载ETF数据...")
print(f"ETF池: {list(ETF_POOL.keys())}")

etf_data = data_loader.load_all_data(
    etf_codes=list(ETF_POOL.keys()),
    start_date=config['start_date'],
    end_date=config['end_date'],
    use_cache=True
)

print(f"\n成功加载 {len(etf_data)} 个ETF数据")

trading_dates = data_loader.get_trading_dates(config['start_date'], config['end_date'])
print(f"交易日数量: {len(trading_dates)}")

In [ ]:
if etf_data:
    sample_code = list(etf_data.keys())[0]
    print(f"\n{sample_code} 数据预览:")
    display(etf_data[sample_code].head())
    print(f"\n数据形状: {etf_data[sample_code].shape}")

## 3. 初始化策略并运行回测

In [ ]:
strategy = ETFRotationStrategy(ETF_POOL, STRATEGY_CONFIG)

backtest = BacktestEngine(
    etf_data_dict=etf_data,
    trading_dates=trading_dates,
    strategy=strategy,
    config=config
)

print("开始回测...")
results = backtest.run()
print("回测完成!")

## 4. 性能分析与可视化

In [ ]:
analyzer = PerformanceAnalyzer(results)

metrics = analyzer.calculate_metrics()

print("=" * 50)
print("策略回测结果")
print("=" * 50)
for key, value in metrics.items():
    print(f"{key}: {value}")
print("=" * 50)

In [ ]:
print("净值曲线:")
analyzer.plot_equity_curve()

In [ ]:
print("回撤曲线:")
analyzer.plot_drawdown()

In [ ]:
print("月度收益率:")
analyzer.plot_monthly_returns()

In [ ]:
print("年度收益率:")
try:
    analyzer.plot_annual_returns()
except Exception as e:
    print(f"绘制年度收益率失败: {e}")

In [ ]:
if not results['trades'].empty:
    print("交易记录预览:")
    display(results['trades'].head(20))
    print(f"\n总交易次数: {len(results['trades'])}")

## 5. 保存报告

In [ ]:
metrics, report_path = analyzer.generate_report()
print(f"报告已保存至: {report_path}")

## 6. 策略参数调优（可选）

可以尝试调整策略参数，观察结果变化。

In [ ]:
import matplotlib.pyplot as plt

test_configs = [
    {'stock_count': 3, 'max_ratio': 0.4},
    {'stock_count': 5, 'max_ratio': 0.3},
    {'stock_count': 7, 'max_ratio': 0.25},
]

print("测试不同配置...")

for i, cfg in enumerate(test_configs):
    print(f"\n配置 {i+1}: {cfg}")
    
    strategy_config = STRATEGY_CONFIG.copy()
    strategy_config.update(cfg)
    
    strategy = ETFRotationStrategy(ETF_POOL, strategy_config)
    backtest = BacktestEngine(etf_data, trading_dates, strategy, config)
    
    results = backtest.run()
    analyzer = PerformanceAnalyzer(results)
    metrics = analyzer.calculate_metrics()
    
    print(f"总收益率: {metrics['总收益率']}%, 夏普比率: {metrics['夏普比率']}, 最大回撤: {metrics['最大回撤']}%")